# Day 13–16 · OOP (Part 2 of 3) — Inheritance & Encapsulation

**Duration:** 50–60 Minutes

### Learning Outcomes
- Understand **inheritance** and its benefit (reuse parent features).
- Use **`super()`** to call the parent's constructor and methods.
- Know the **5 types** of inheritance: single, multilevel, hierarchical, multiple, hybrid.
- Understand **MRO** (Method Resolution Order) and **method overriding**.
- Use **`isinstance()`** and **`issubclass()`**.
- Apply **encapsulation**: public / protected / private, name mangling.
- Use **getters/setters** and the pythonic **`@property`**.

## 1. What is Inheritance?

**Inheritance** lets a new class (**child / subclass**) reuse the attributes and methods of an
existing class (**parent / base / superclass**). The child gets everything the parent has, and can
add or change behaviour.

```text
        Parent (base)  -- name, age, greet()
             |
             v  inherits
        Child (derived) -- + student_id, + own greet()
```

### Terminology
| Term | Meaning |
|---|---|
| Parent / Base / Super class | The class being inherited **from** |
| Child / Derived / Sub class | The class that **inherits** |
| `super()` | A reference to the parent class |

**Why inheritance?** Reuse code, avoid duplication, and model "**is-a**" relationships
(a Student **is a** Person).

## 2. Basic Inheritance

**Syntax**
```python
class Child(Parent):
    ...
```

The child automatically has the parent's methods and attributes.

In [1]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def greet(self):
        return f"Hello, my name is {self.name} and I am {self.age} years old."

class Student(Person):        # Student inherits from Person
    pass                      # no new code yet -> gets everything from Person

s = Student("Alice", 22)
print(s.greet())              # inherited method works
print(s.name, s.age)          # inherited attributes

Hello, my name is Alice and I am 22 years old.
Alice 22


## 3. Extending the Child with `super()`

Usually the child adds its **own** data. Call the parent's `__init__` with **`super()`** so you do
not repeat the parent's setup.

**Key Notes:**
- `super().__init__(...)` runs the parent constructor.
- The older style `Person.__init__(self, ...)` also works, but `super()` is preferred.

In [2]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def greet(self):
        return f"Hi, I'm {self.name}, age {self.age}."

class Student(Person):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)     # reuse parent's setup
        self.student_id = student_id     # add child's own data

    def show_id(self):
        return f"{self.name}'s ID is {self.student_id}"

s = Student("Alice", 22, "S1234")
print(s.greet())      # inherited from Person
print(s.show_id())    # defined in Student

Hi, I'm Alice, age 22.
Alice's ID is S1234


## 4. The 5 Types of Inheritance

```text
1. SINGLE              2. MULTILEVEL         3. HIERARCHICAL
   A                      A                      A
   |                      |                    /   \
   B                      B                   B     C
                          |
                          C

4. MULTIPLE            5. HYBRID (mix of the above, e.g. a diamond)
   A   B                     A
    \ /                     / \
     C                     B   C
                            \ /
                             D
```

| Type | Meaning |
|---|---|
| **Single** | One child inherits one parent |
| **Multilevel** | Chain: grandparent → parent → child |
| **Hierarchical** | Many children share one parent |
| **Multiple** | One child inherits from several parents |
| **Hybrid** | A combination of the above |

## 5. Single Inheritance

One child, one parent (the example in Section 3). Simplest and most common.

In [3]:
class Vehicle:
    def __init__(self, brand):
        self.brand = brand
    def info(self):
        return f"Vehicle brand: {self.brand}"

class Car(Vehicle):                 # single inheritance
    def wheels(self):
        return "Has 4 wheels"

c = Car("Toyota")
print(c.info())        # from parent
print(c.wheels())      # from child

Vehicle brand: Toyota
Has 4 wheels


## 6. Multilevel Inheritance

A chain: `Person → Adult → Retired`. Each level inherits from the one above it.

In [4]:
class Person:
    def __init__(self, name):
        self.name = name

class Adult(Person):                # level 2
    def __init__(self, name, job):
        super().__init__(name)
        self.job = job

class Retired(Adult):               # level 3 (inherits Adult -> Person)
    def __init__(self, name, job, retire_age):
        super().__init__(name, job)
        self.retire_age = retire_age
    def info(self):
        return f"{self.name}, ex-{self.job}, retired at {self.retire_age}"

r = Retired("Mary", "Teacher", 60)
print(r.info())
print(r.name)          # reaches all the way up to Person

Mary, ex-Teacher, retired at 60
Mary


## 7. Hierarchical Inheritance

Several children share **one** parent. Here `Student` and `Employee` both inherit `Person`.

In [5]:
class Person:
    def __init__(self, name):
        self.name = name
    def greet(self):
        return f"Hi, I'm {self.name}."

class Student(Person):
    def role(self):
        return f"{self.name} is a student."

class Employee(Person):
    def role(self):
        return f"{self.name} is an employee."

s = Student("Alice")
e = Employee("Bob")
print(s.greet(), "|", s.role())
print(e.greet(), "|", e.role())

Hi, I'm Alice. | Alice is a student.
Hi, I'm Bob. | Bob is an employee.


## 8. Multiple Inheritance

One child inherits from **several** parents and gets features from all of them.

In [6]:
class Engine:
    def start(self):
        return "Engine started"

class Radio:
    def play(self):
        return "Playing music"

class Car(Engine, Radio):           # inherits from BOTH
    def drive(self):
        return "Car is driving"

c = Car()
print(c.start())     # from Engine
print(c.play())      # from Radio
print(c.drive())     # from Car

Engine started
Playing music
Car is driving


## 9. Method Resolution Order (MRO)

With multiple inheritance, Python needs rules to decide **which parent's method wins**. The
**MRO** is the order Python searches: left-to-right, children before parents. View it with
`ClassName.__mro__` or `ClassName.mro()`.

**Key Notes:**
- Python uses the **C3 linearization** algorithm to build the MRO.
- The first match in the MRO is used — so the **leftmost** parent takes priority.

In [7]:
class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B"

class C(A):
    def who(self):
        return "C"

class D(B, C):        # diamond shape
    pass

d = D()
print(d.who())        # 'B'  -> B comes before C in the MRO
print([cls.__name__ for cls in D.__mro__])   # D, B, C, A, object

B
['D', 'B', 'C', 'A', 'object']


## 10. Method Overriding

A child can **redefine** a method it inherited, giving its own version. When you call the method
on a child object, the **child's** version runs.

In [8]:
class Person:
    def greet(self):
        return "Hello from a person."

class Student(Person):
    def greet(self):                    # OVERRIDES Person.greet
        return "Hello from a student."

print(Person().greet())     # Hello from a person.
print(Student().greet())    # Hello from a student.  (child wins)

Hello from a person.
Hello from a student.


## 11. Extending (not replacing) with `super()`

Inside an overridden method you can still call the parent's version with `super()` and add to it.

In [9]:
class Person:
    def greet(self):
        return "Hi, I'm a person"

class Student(Person):
    def greet(self):
        base = super().greet()          # reuse parent's greeting
        return base + " and also a student"

print(Student().greet())    # Hi, I'm a person and also a student

Hi, I'm a person and also a student


## 12. `isinstance()` and `issubclass()`

- `isinstance(obj, Class)` → is `obj` an instance of `Class` (or a subclass)?
- `issubclass(Child, Parent)` → is one class derived from another?

In [10]:
class Person: pass
class Student(Person): pass

s = Student()
print(isinstance(s, Student))    # True
print(isinstance(s, Person))     # True  (Student IS-A Person)
print(isinstance(s, str))        # False
print(issubclass(Student, Person))  # True
print(issubclass(Person, Student))  # False

True
True
False
True
False


## 13. Encapsulation

**Encapsulation** = bundling data and the methods that work on it inside one class, and
**controlling access** to that data. It protects an object's internal state from accidental
changes and lets you validate updates.

### Access levels (by convention)
| Level | Prefix | Meaning |
|---|---|---|
| **Public** | `name` | Accessible everywhere (default) |
| **Protected** | `_name` | "Internal — please don't touch" (convention only) |
| **Private** | `__name` | Name-mangled; not meant to be accessed from outside |

**Key Notes:**
- Python has **no true `private`** like Java/C++ — these are conventions + name mangling.
- `_single` = a *hint*; `__double` triggers **name mangling** (`_ClassName__name`).

## 14. Public, Protected, Private in Action

In [11]:
class Account:
    def __init__(self):
        self.owner = "Alice"       # public
        self._bank = "SBI"         # protected (convention)
        self.__balance = 1000      # private (name-mangled)

a = Account()
print(a.owner)      # public   -> OK
print(a._bank)      # protected -> works, but "hands off"

# Private attribute is NOT directly accessible by its written name:
try:
    print(a.__balance)
except AttributeError as e:
    print("cannot access __balance directly:", e)

# It still exists under a mangled name (shown for understanding, not for real use):
print("mangled access:", a._Account__balance)

Alice
SBI
cannot access __balance directly: 'Account' object has no attribute '__balance'
mangled access: 1000


## 15. Getters and Setters

To use private data safely, expose **methods** that read (getter) and update (setter) it — the
setter can **validate** the new value.

In [12]:
class Account:
    def __init__(self, balance=0):
        self.__balance = balance          # private

    def get_balance(self):                # getter
        return self.__balance

    def set_balance(self, amount):        # setter with validation
        if amount < 0:
            print("balance cannot be negative")
            return
        self.__balance = amount

acc = Account(1000)
print(acc.get_balance())     # 1000
acc.set_balance(-50)         # rejected
acc.set_balance(2000)        # accepted
print(acc.get_balance())     # 2000

1000
balance cannot be negative
2000


## 16. The Pythonic Way: `@property`

Getters/setters are clunky to call (`obj.get_x()`). The **`@property`** decorator lets you use
simple attribute syntax (`obj.x`) while still running validation behind the scenes.

**Key Notes:**
- `@property` makes a method act like a **read-only attribute**.
- `@<name>.setter` adds a settable version with validation.

In [13]:
class Circle:
    def __init__(self, radius):
        self._radius = radius             # "protected" storage

    @property
    def radius(self):                     # getter -> use as c.radius
        return self._radius

    @radius.setter
    def radius(self, value):              # setter -> use as c.radius = 10
        if value < 0:
            raise ValueError("radius cannot be negative")
        self._radius = value

    @property
    def area(self):                       # computed, read-only
        return round(3.14159 * self._radius ** 2, 2)

c = Circle(5)
print(c.radius)     # 5    (looks like an attribute, runs the getter)
print(c.area)       # 78.54 (computed on the fly)
c.radius = 10       # runs the setter
print(c.area)       # 314.16

try:
    c.radius = -3   # setter rejects it
except ValueError as e:
    print("rejected:", e)

5
78.54
314.16
rejected: radius cannot be negative


## 17. Common Mistakes

**1. Forgetting `super().__init__()`** — the child then misses the parent's setup, so parent
attributes never get created.

**2. Thinking `__private` is truly hidden** — it's only name-mangled; `_Class__private` still
reaches it. Treat it as "don't touch", not "impossible to touch".

**3. Wrong parent order in multiple inheritance** — order matters for the MRO; the **leftmost**
parent wins on clashes.

**4. Re-writing the parent's method entirely when you meant to extend it** — call `super()` inside
the override to keep the parent behaviour.

## 18. Mini Example — Encapsulated `BankAccount`

Combines inheritance-friendly design with encapsulation and a `@property`.

In [14]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance          # private

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount > self.__balance:
            raise ValueError("insufficient funds")
        self.__balance -= amount

class SavingsAccount(BankAccount):        # inherits everything
    def __init__(self, owner, balance=0, rate=0.04):
        super().__init__(owner, balance)
        self.rate = rate

    def add_interest(self):
        self.deposit(self.balance * self.rate)

acc = SavingsAccount("Alice", 1000)
acc.deposit(500)
acc.add_interest()
print(acc.owner, "->", round(acc.balance, 2))

try:
    acc.withdraw(999999)
except ValueError as e:
    print("blocked:", e)

Alice -> 1560.0
blocked: insufficient funds


## 19. Summary

- **Inheritance** lets a child reuse a parent's features (an "**is-a**" relationship).
- Use **`super().__init__(...)`** to reuse the parent constructor.
- Five types: **single, multilevel, hierarchical, multiple, hybrid**.
- **MRO** decides which method wins in multiple inheritance (leftmost parent first).
- **Method overriding** lets a child replace/extend an inherited method.
- **Encapsulation** hides internal data: `public`, `_protected`, `__private` (name-mangled).
- Prefer **`@property`** for clean, validated attribute access.

## 20. Practice Questions

1. Create a `Person` base class and a `Teacher` subclass that adds a `subject`.
2. Use `super().__init__()` in the subclass constructor.
3. Build a multilevel chain: `Animal → Dog → Puppy`.
4. Build hierarchical inheritance: `Shape → Circle` and `Shape → Square`.
5. Create multiple inheritance: a `SmartPhone(Phone, Camera)` class.
6. Print the `__mro__` of your multiple-inheritance class and explain the order.
7. Override a parent method in a child and call the parent version with `super()`.
8. Make a `Student` class with a private `__marks` and getter/setter methods.
9. Rewrite that `Student` using `@property` with validation (marks 0–100).
10. Use `isinstance()` and `issubclass()` to test your class relationships.